# Fallback: open model extraction on Colab

수업용 API 서버가 안 될 때만 사용합니다. Colab 안에서 작은 공개 instruction model을 직접 실행해서 `case_issues.jsonl` 또는 `issue_features.jsonl`을 만듭니다.

품질과 속도는 `gpt-5.4-mini`보다 떨어질 수 있습니다. 수업 중에는 `LIMIT`을 작게 두고 구조를 이해하는 용도로 쓰세요.

In [ ]:
!pip -q install transformers accelerate bitsandbytes pydantic


In [ ]:
from pathlib import Path

# Colab 파일 패널에 업로드했으면 /content 아래 경로를 사용합니다.
# Google Drive를 mount했다면 /content/drive/MyDrive/... 형태로 바꿔도 됩니다.
PDF_PAGES_PATH = Path('/content/pdf_pages.jsonl')
CASE_ISSUES_PATH = Path('/content/case_issues.jsonl')
CASE_ISSUES_OUTPUT = Path('/content/case_issues_fallback.jsonl')
ISSUE_FEATURES_OUTPUT = Path('/content/issue_features_fallback.jsonl')

LIMIT = 2
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'


In [ ]:
import json, re
from collections import defaultdict
from pydantic import BaseModel, Field

def load_jsonl(path):
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def extract_json_object(text):
    text = re.sub(r'^\s*```(?:json|JSON)?\s*\n?', '', text.strip())
    text = re.sub(r'\n?\s*```\s*$', '', text).strip()
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError(text[:300])
    return json.loads(text[start:end + 1])

def group_pages(rows):
    docs = defaultdict(list)
    for row in rows:
        filename = row.get('metadata', {}).get('filename')
        if filename:
            docs[filename].append(row)
    for pages in docs.values():
        pages.sort(key=lambda x: x.get('metadata', {}).get('page', 0))
    return dict(docs)

def merge_pages(pages):
    parts = []
    for page in pages:
        page_no = page.get('metadata', {}).get('page', '?')
        content = (page.get('page_content') or '').strip()
        if content:
            parts.append(f'[페이지 {page_no}]\n{content}')
    return '\n\n'.join(parts)

def iter_issue_records(rows):
    for row in rows:
        document_id = row.get('document_id', '')
        for issue_index, issue in enumerate(row.get('issues', []) or []):
            yield {
                'document_id': document_id,
                'issue_index': issue_index,
                'case_title': row.get('case_title', ''),
                'decision_type': row.get('decision_type', ''),
                'tax_item': row.get('tax_item', ''),
                'issue': issue.get('issue', ''),
                'taxpayer_argument': issue.get('taxpayer_argument', ''),
                'judgment_reasoning': issue.get('judgment_reasoning', ''),
                'conclusion': issue.get('conclusion', ''),
            }


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
quant_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    quantization_config=quant_config,
)

def ask_json(system_prompt, user_payload, max_new_tokens=1200):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_payload},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return extract_json_object(text)


## 1. PDF page text -> case issues

`pdf_pages.jsonl`을 업로드한 경우 실행합니다. 출력은 `/content/case_issues_fallback.jsonl`입니다.

In [ ]:
ISSUE_SYSTEM = '''당신은 한국 조세심판례를 분석하는 전문가입니다.
원문을 읽고 다음 JSON 객체만 출력하세요.
{
  "case_title": "사건 제목 또는 핵심 주제",
  "decision_type": "인용/기각/일부인용/각하 등",
  "tax_item": "세목",
  "issues": [
    {
      "issue": "쟁점 한 문장",
      "taxpayer_argument": "청구인 핵심 주장",
      "judgment_reasoning": "판단 근거",
      "conclusion": "결론"
    }
  ]
}
JSON 이외의 설명은 쓰지 마세요.'''

if PDF_PAGES_PATH.exists():
    docs = list(group_pages(load_jsonl(PDF_PAGES_PATH)).items())[:LIMIT]
    with CASE_ISSUES_OUTPUT.open('w', encoding='utf-8') as f:
        for filename, pages in docs:
            obj = ask_json(ISSUE_SYSTEM, merge_pages(pages)[:24000])
            obj['document_id'] = filename
            f.write(json.dumps(obj, ensure_ascii=False) + '\n')
            print('ok', filename)
    print(CASE_ISSUES_OUTPUT)
else:
    print('skip: pdf_pages.jsonl not found')


## 2. Case issues -> graph features

`case_issues.jsonl` 또는 위에서 만든 `case_issues_fallback.jsonl`을 사용합니다. 출력은 `/content/issue_features_fallback.jsonl`입니다.

In [ ]:
FEATURE_SYSTEM = '''당신은 한국 조세심판례를 graph retrieval용 feature로 구조화하는 전문가입니다.
입력으로 주어진 하나의 쟁점 record에서 다음 JSON 객체만 출력하세요.
{
  "legal_concepts": ["법적/세무적 개념 phrase"],
  "fact_patterns": ["판단에 영향을 준 사실관계 패턴 phrase"],
  "evidence_types": ["증빙 종류 phrase"],
  "outcome": "기각/인용/일부인용/재조사/각하 등"
}
아무 entity나 뽑지 말고 위 네 종류만 추출하세요. JSON 이외의 설명은 쓰지 마세요.'''

source_path = CASE_ISSUES_PATH if CASE_ISSUES_PATH.exists() else CASE_ISSUES_OUTPUT
if source_path.exists():
    records = list(iter_issue_records(load_jsonl(source_path)))[:LIMIT]
    with ISSUE_FEATURES_OUTPUT.open('w', encoding='utf-8') as f:
        for record in records:
            payload = json.dumps(record, ensure_ascii=False, indent=2)
            features = ask_json(FEATURE_SYSTEM, payload, max_new_tokens=700)
            f.write(json.dumps({**record, **features}, ensure_ascii=False) + '\n')
            print('ok', record['document_id'], record['issue_index'])
    print(ISSUE_FEATURES_OUTPUT)
else:
    print('skip: case_issues.jsonl not found')
